## Overview

CNN-GRU Setup for prediction out of 2D timeseries data

finding out of impact of training on "wrong" year of the saison

CoPilot Hyperparametertuned

Colab version of local, loading data from existing dataloaders

----
## Data:

-2D Space - Timeseries

-predicting 1 feature out of 7 variables

-Forecasting 1 timestep (not the following)

-------
Peter Resch, 6.6.

In [ ]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
import pandas as pd
from pandas import Series, DataFrame
from matplotlib import pyplot as plt
import numpy as np
import os
import random

from itertools import product
from pathlib import Path

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, ConcatDataset # Added ConcatDataset
import sklearn
from sklearn.model_selection import train_test_split

import xarray as xr
rSeed=42

SavePlots = False

## Loading Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

import sys
sys.path.insert(0, '/content/drive/MyDrive/small_grid/') # Add the directory containing the module to the Python path

import my_dataloader_module

In [ ]:
!cd drive/MyDrive/small_grid && ls

In [ ]:
seasons={"spring": "MAM", "summer": "JJA", "autumn": "SON", "winter": "DJF"}

data_dir='drive/MyDrive/small_grid'
!cd {data_dir} && ls

dataloader_path = str(data_dir+ "/dataloaders/")

dataloaderlist = [p.name for p in Path(dataloader_path).iterdir() if p.is_file()]
#clean dataloaderlist, let only .pt files
dataloaderlist = [f for f in dataloaderlist if f.endswith('.pt')]
dataloaderlist=[f for f in dataloaderlist if "dataloader" in f]      #select only the ones with "dataloader" in the name

## Loading Dataloaders

In [ ]:
dataloaders = {}
i=0
for dl in dataloaderlist:
    time,aim = dl.split("_dataloader_")
    time,season=time.split("_")
    aim = aim.split(".pt")[0]
    #print(time, season, aim)
    dataloaders[time,season,aim] = torch.load(dataloader_path + time + "_" + season + "_dataloader_"+ aim+".pt", weights_only=False)
    i=i+1
print(i,"Dataloaders loaded successfully.")

## Building the Neural Network

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


hidden_size=16

class CNN_GRU(nn.Module):
    def __init__(self,in_channels=7,gru_hidden_size=256,num_layers=1,dropout=0.2,lat_size=5,lon_size=5):
        super().__init__()

        #compressing space
        def double_conv(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True)
            )
        self.conv1 = double_conv(in_channels, 16)
        self.maxpool = nn.MaxPool2d(kernel_size=(2, 2))
        self.conv2 = double_conv(16, 32)
        self.lat_size = lat_size
        self.lon_size = lon_size

        #compressing time
        gru_dropout = dropout if num_layers > 1 else 0.0
        self.gru = nn.GRU(input_size=32, hidden_size=gru_hidden_size, num_layers=num_layers, dropout=gru_dropout, batch_first=True)
        self.projection = nn.Linear(gru_hidden_size, 32 * lat_size * lon_size)

        #expanding space
        self.deconv1 = nn.ConvTranspose2d(32, 16, kernel_size=3, padding=1)
        self.deconv2 = nn.ConvTranspose2d(16, 1, kernel_size=3, padding=1)


    def forward(self, x):
        #print("starting forward:",x.shape)
        x_seq=[]
        #recognize patterns of the spatial data with CNN
        for time in range(x.shape[1]):
            #print(time)
            cnn_in=x[:, time, :, :]
            #print("time,cnn_in.shape:",time,cnn_in.shape)
            cnn_in=self.conv1(cnn_in)
            #print("after conv1:",cnn_in.shape)
            cnn_in = self.maxpool(cnn_in)
            #print("after maxpool:",cnn_in.shape)
            cnn_in = self.conv2(cnn_in)
            #print("after conv2:",cnn_in.shape)
            cnn_in = self.maxpool(cnn_in)
            #print("after maxpool:",cnn_in.shape)
            cnn_in = cnn_in.view(cnn_in.size(0), -1)  # Flatten for GRU input
            #print("after flatten:",cnn_in.shape)
            x_seq.append(cnn_in)
        x_seq = torch.stack(x_seq, dim=1)
        #print("shaped for GRU:",x_seq.shape)#(batch, time, convoluted features with space)

        #decoding the temporal patterns with GRU
        x, hn = self.gru(x_seq)#x:(batch, time, hidden_size)
        x = self.projection(x[:,-1,:]).view(x.shape[0],32,self.lat_size,self.lon_size)##(batch, last hidden_size,lat_size,lon_size)
        #print("after GRU:",x.shape)
        x=self.deconv1(x)
        #print("after deconv1:",x.shape)
        x=self.deconv2(x)
        #print("after deconv2:",x.shape)
        return x#


cnn_gru_model=CNN_GRU().to(device)
print(cnn_gru_model)


In [ ]:
def train(dataloader, model, loss_fn, optimizer,device):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        #print(X.shape, y.shape)
        X, y = X.to(device), y.to(device)
        #print(X.shape, y.shape)
        pred = model(X)#.squeeze()
        #print(pred)#.shape)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            #print(f"Train Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    return loss.item()



def test(dataloader, model, loss_fn,device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            predictions = model(X)
            test_loss += loss_fn(predictions, y).item()

            all_predictions.append(predictions.detach().cpu().numpy())
            all_targets.append(y.detach().cpu().numpy())

    test_loss /= num_batches
    y_pred = np.concatenate(all_predictions)#[:,:,0]
    y_true = np.concatenate(all_targets)#[:,:,0]
    #print(f"y_true shape: {y_true.shape}, y_pred shape: {y_pred.shape}")

    # Flatten everything to 2D: (samples, features)
    y_true_flat = y_true.reshape(y_true.shape[0], -1)
    y_pred_flat = y_pred.reshape(y_pred.shape[0], -1)

    #mae = sklearn.metrics.mean_absolute_error(y_true, y_pred)
    #rmse = np.sqrt(sklearn.metrics.mean_squared_error(y_true, y_pred))
    r2 = sklearn.metrics.r2_score(y_true_flat, y_pred_flat)
    print(f"Test Error:\n R2: {r2:>8f}, Avg loss: {test_loss:>8f} \n")
    return test_loss

## Train the model

In [ ]:
# Original line: time = "recent"

# Define the time categories to combine
times_to_combine = ["recent", "past"]
combined_time_tag = "_and_".join(times_to_combine)

# Dictionary to hold the combined dataloaders
combined_dataloaders_dict = {}

# Iterate over each season to create combined train and validation dataloaders
for season_key in seasons.keys():
    for aim_key in ["train", "val"]:
        datasets_to_concat = []
        for time_key in times_to_combine:
            current_dataloader = dataloaders.get((time_key, season_key, aim_key))
            if current_dataloader:
                datasets_to_concat.append(current_dataloader.dataset)
            else:
                print(f"Warning: Dataloader not found for {(time_key, season_key, aim_key)}. Skipping.")

        if datasets_to_concat:
            combined_dataset = ConcatDataset(datasets_to_concat)

            # Attempt to get DataLoader properties from the first available dataloader
            first_dataloader = dataloaders.get((times_to_combine[0], season_key, aim_key))
            if first_dataloader:
                batch_size = first_dataloader.batch_size
                shuffle = (aim_key == "train") # Typically only shuffle training data
                num_workers = getattr(first_dataloader, 'num_workers', 0) # Default to 0 if not present

                combined_dataloaders_dict[(combined_time_tag, season_key, aim_key)] = DataLoader(
                    combined_dataset,
                    batch_size=batch_size,
                    shuffle=shuffle,
                    num_workers=num_workers
                )
            else:
                print(f"Error: Could not determine DataLoader properties for {(combined_time_tag, season_key, aim_key)}. Skipping combination.")
        else:
            print(f"No datasets to combine for {(combined_time_tag, season_key, aim_key)}. Skipping.")

# Set 'time' variable to the combined tag for use in model naming and dictionary lookups
time = combined_time_tag

epochs = 50
tuning_epochs = 12
early_stop_patience = 6
min_delta = 1e-4
hyperparam_trials = [
    {"learning_rate": 1e-4, "weight_decay": 0.0, "gru_hidden_size": 128, "dropout": 0.0, "num_layers": 1},
    {"learning_rate": 3e-4, "weight_decay": 1e-5, "gru_hidden_size": 128, "dropout": 0.2, "num_layers": 1},
    {"learning_rate": 1e-3, "weight_decay": 1e-4, "gru_hidden_size": 256, "dropout": 0.2, "num_layers": 1},
    {"learning_rate": 3e-4, "weight_decay": 1e-4, "gru_hidden_size": 256, "dropout": 0.4, "num_layers": 2},
    {"learning_rate": 1e-4, "weight_decay": 1e-4, "gru_hidden_size": 512, "dropout": 0.2, "num_layers": 2},
    {"learning_rate": 3e-4, "weight_decay": 0.0, "gru_hidden_size": 512, "dropout": 0.4, "num_layers": 1},
]

def run_training_loop(model, train_loader, val_loader, loss_fcn, optimizer, scheduler, device, epochs, early_stop_patience, min_delta, test_loader=None):
    best_val_loss = float("inf")
    best_epoch = 0
    best_state_dict = None
    epochs_without_improvement = 0
    train_losses = []
    val_losses = []
    test_losses = []

    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")
        train_loss = train(train_loader, model, loss_fcn, optimizer, device)
        val_loss = test(val_loader, model, loss_fcn, device)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if test_loader is not None:
            test_loss = test(test_loader, model, loss_fcn, device)
            test_losses.append(test_loss)

        scheduler.step(val_loss)

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_epoch = t + 1
            epochs_without_improvement = 0
            best_state_dict = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stop_patience:
            print(
                f"Early stopping triggered at epoch {t+1}. "
                f"Best validation loss: {best_val_loss:>8f} at epoch {best_epoch}."
            )
            break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return {
        "model": model,
        "train_losses": np.array(train_losses),
        "val_losses": np.array(val_losses),
        "test_losses": np.array(test_losses),
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch
    }


for season in seasons.keys():
    print(":"*50)
    print(f"Season: {season}")
    model_name = f"cnn_gru_tuned_{time}_{season}"

    tuning_results = []
    for trial_index, trial_params in enumerate(hyperparam_trials, start=1):
        print(f"Tuning trial {trial_index}/{len(hyperparam_trials)}: {trial_params}")
        trial_model = CNN_GRU(**{
            "gru_hidden_size": trial_params["gru_hidden_size"],
            "num_layers": trial_params["num_layers"],
            "dropout": trial_params["dropout"],
        }).to(device)
        loss_fcn = nn.MSELoss()
        optimizer = AdamW(trial_model.parameters(), lr=trial_params["learning_rate"], weight_decay=trial_params["weight_decay"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6)

        tuning_run = run_training_loop(
            trial_model,
            combined_dataloaders_dict[(time, season, "train")], # Changed to combined dataloaders
            combined_dataloaders_dict[(time, season, "val")],   # Changed to combined dataloaders
            loss_fcn,
            optimizer,
            scheduler,
            device,
            tuning_epochs,
            early_stop_patience,
            min_delta,
            test_loader=None
        )
        tuning_results.append({
            **trial_params,
            "trial": trial_index,
            "best_val_loss": tuning_run["best_val_loss"],
            "best_epoch": tuning_run["best_epoch"]
        })

    tuning_df = pd.DataFrame(tuning_results).sort_values(["best_val_loss", "best_epoch"]).reset_index(drop=True)
    tuning_df.to_csv(f"{data_dir}/models/tuning_{model_name}.csv", index=False)

    best_params = tuning_df.iloc[0].to_dict()
    print(f"Best tuned parameters for {season}: {best_params}")

    loss_fcn = nn.MSELoss()
    cnn_gru_model = CNN_LSTM(**{
        "gru_hidden_size": int(best_params["gru_hidden_size"]),
        "num_layers": int(best_params["num_layers"]),
        "dropout": float(best_params["dropout"]),
    }).to(device)
    optimizer_cnn_gru = AdamW(cnn_gru_model.parameters(), lr=float(best_params["learning_rate"]), weight_decay=float(best_params["weight_decay"]))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_cnn_gru, mode="min", factor=0.5, patience=2, min_lr=1e-6)

    final_run = run_training_loop(
        cnn_gru_model,
        combined_dataloaders_dict[(time, season, "train")], # Changed to combined dataloaders
        combined_dataloaders_dict[(time, season, "val")],   # Changed to combined dataloaders
        loss_fcn,
        optimizer_cnn_gru,
        scheduler,
        device,
        epochs,
        early_stop_patience,
        min_delta,
        test_loader=dataloaders["now", season, "test"]
    )

    train_losses = final_run["train_losses"]
    val_losses = final_run["val_losses"]
    test_losses = final_run["test_losses"]
    best_val_loss = final_run["best_val_loss"]
    best_epoch = final_run["best_epoch"]

    print("Training done!")

    os.makedirs("models", exist_ok=True)
    save_path = f"{data_dir}/models/{model_name}.pth"

    torch.save({
        "model_state_dict": cnn_gru_model.state_dict(),
        "optimizer_state_dict": optimizer_cnn_gru.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "lag_selection": [0, 24, 48, 60, 66, 69, 71],
        "target_var": 4,
        "epoch": best_epoch if best_epoch > 0 else len(train_losses),
        "max_epochs": epochs,
        "best_val_loss": best_val_loss,
        "early_stop_patience": early_stop_patience,
        "min_delta": min_delta,
        "best_hyperparameters": {
            "learning_rate": float(best_params["learning_rate"]),
            "weight_decay": float(best_params["weight_decay"]),
            "gru_hidden_size": int(best_params["gru_hidden_size"]),
            "dropout": float(best_params["dropout"]),
            "num_layers": int(best_params["num_layers"])
        }
    }, save_path)

    print(f"Saved model to {save_path}")

    df = pd.DataFrame({
        "Train Loss": train_losses,
        "Validate Loss": val_losses,
        "Test Loss": test_losses
    })
    df.to_csv(f"{data_dir}/models/losses_{model_name}.csv", index=False)
    print(f"Saved losses to losses_{model_name}.csv")

    print(f"Finished traininig {model_name} for {len(train_losses)} epochs.")
    print("-"*30)

::::::::::::::::::::::::::::::::::::::::::::::::::
Season: spring
Tuning trial 1/6: {'learning_rate': 0.0001, 'weight_decay': 0.0, 'lstm_hidden_size': 128, 'dropout': 0.0, 'num_layers': 1}
Epoch 1
-------------------------------
Test Error:
 R2: 0.400626, Avg loss: 0.668446 

Epoch 2
-------------------------------
Test Error:
 R2: 0.531985, Avg loss: 0.523113 

Epoch 3
-------------------------------
Test Error:
 R2: 0.482480, Avg loss: 0.576271 

Epoch 4
-------------------------------
Test Error:
 R2: 0.566320, Avg loss: 0.484186 

Epoch 5
-------------------------------
Test Error:
 R2: 0.604077, Avg loss: 0.442556 

Epoch 6
-------------------------------
Test Error:
 R2: 0.602399, Avg loss: 0.444550 

Epoch 7
-------------------------------
Test Error:
 R2: 0.601092, Avg loss: 0.445300 

Epoch 8
-------------------------------
Test Error:
 R2: 0.607696, Avg loss: 0.438123 

Epoch 9
-------------------------------
Test Error:
 R2: 0.616945, Avg loss: 0.427554 

Epoch 10
----------